# tools

> The tools, written against `Host` instead of against an IDE.

The point has not changed from the version this grew out of: the model gets the *same*
powers the person sitting in front of leela has, through the same code paths -- see the
code (an index that covers the repo *and* every installed package), read the web, edit
files by hash-verified address, run code in the live namespace. What changed is that none
of it now knows what a `Workspace` is.

Three invariants hold across all of it, and they are why this file is boring:

*Every path goes through `Host.check`.* exhash and fossick both write to disk on their own
account; each is handed a path the host has already resolved and approved, so no tool can
address anything outside the open folders.

*Every tool returns a string, clipped here.* Tool results go straight back into the
context window, so truncation happens before the tokens are spent rather than after.

*A capability the host does not have is not offered.* `tools_for` builds the list by
trying each group and dropping the ones that raise `NotImplementedError`. Telling a model
about a tool that always fails is worse than never mentioning it: it will keep trying.


In [ ]:
#| default_exp tools

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import json
from ramabana.core import agent_err

In [ ]:
#| export
MAX_TOOL_CHARS = 6000     # per tool result; the context window is the scarce resource here

In [ ]:
#| export
MAX_HITS = 20

In [ ]:
#| export
# The tools that change something on disk or in the live session. Named as a set because
# that is the line an approval policy needs to draw -- see `hitl.Approvals`.
WRITE_TOOLS = frozenset({'edit_file', 'create_file', 'edit_cell', 'add_cell', 'run_python'})

In [ ]:
#| export
def clip(s, n=MAX_TOOL_CHARS):
    s = str(s)
    return s if len(s) <= n else s[:n] + f'\n…[{len(s)-n} more chars]'

In [ ]:
#| export
def _cmds(commands):
    """Parse exhash commands from what a tool call can carry.

    Models emit JSON, exhash wants tuples: `[["12|a1b2|","s","old","new"]]` becomes
    `[("12|a1b2|","s","old","new")]`. Nested command tuples (`g`/`v`) recurse.
    """
    if isinstance(commands, str): commands = json.loads(commands)
    if not isinstance(commands, list): raise ValueError('commands must be a JSON list of command arrays')
    def _t(c):
        if not isinstance(c, (list, tuple)): raise ValueError(f'each command must be an array, got {type(c).__name__}')
        return tuple(_t(x) if isinstance(x, (list, tuple)) else x for x in c)
    return [_t(c) for c in commands]

In [ ]:
#| export
def _probe(host, *calls):
    "Whether every one of `calls` is supported. A host says 'no' by raising `NotImplementedError`."
    for f in calls:
        try: f()
        except NotImplementedError: return False
        except Exception: pass
    return True

In [ ]:
#| export
# ---------------------------------------------------------------------------
def code_tools(host):
    "Seeing the code: the index, the shapes in it, and the files it covers."

    def search_code(query: str) -> str:
        """Search the codebase and every installed package for `query`.

        Semantic when the code index is built, a literal scan otherwise. Use this before
        writing anything non-trivial: the answer is usually already in the environment.
        """
        hits = host.search(query, limit=MAX_HITS)
        if not hits: return f'no matches ({host.search_note})'
        rows = [f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}' for h in hits]
        return clip(f'[{host.search_note}]\n' + '\n'.join(rows))

    def similar_code(path: str, line: int = 1) -> str:
        "Find code shaped like the function at `path`:`line` -- every place a pattern was already used."
        hits = host.peers(str(host.check(path)), int(line), limit=MAX_HITS)
        if not hits: return f'nothing similar ({host.search_note})'
        return clip('\n'.join(f'{h.path}:{h.line}  {h.symbol or ""}  {h.text}' for h in hits))

    def outline(path: str) -> str:
        "The defs and classes in one file, with line numbers."
        syms = host.symbols(str(host.check(path)))
        if not syms: return f'no symbols in {path}'
        return clip('\n'.join(f'{int(getattr(s, "score", 0))*" "}{s.line}: {s.symbol}' for s in syms))

    def list_files(pattern: str = '') -> str:
        "Files in the open folders, optionally filtered by a substring of the path."
        ps = [str(p) for p in host.walk()]
        if pattern: ps = [p for p in ps if pattern.lower() in p.lower()]
        return clip('\n'.join(ps[:400]) or 'no matching files')

    return [search_code, similar_code, outline, list_files]

In [ ]:
#| export
def file_tools(host):
    "Reading and editing files, always by hash-verified address."

    def view_file(path: str, start: int = 0, end: int = 0) -> str:
        """Read a file as `lineno|hash|content` lines. Optionally limit to lines `start`..`end`.

        Always read this way before editing: `edit_file` addresses lines by the exact
        hashes this returns, so the view is also the address book.
        """
        from exhash import lnhashview_file
        p = host.check(path)
        if not p.exists(): return f'no such file: {p}'
        return clip(str(lnhashview_file(str(p), start or None, end or None)))

    def edit_file(path: str, commands: str) -> str:
        """Edit a file with hash-verified exhash commands, and return the diff.

        `commands` is a JSON array of command arrays, each starting with an address taken
        from `view_file`, e.g.
          [["12|a1b2|", "s", "old text", "new text"],
           ["30|9f3c|", "a", "a new line appended after line 30"]]
        Every address's hash is checked immediately before it runs, so an edit built on a
        stale view fails instead of damaging the wrong line. Nothing is written unless
        every command succeeds.
        """
        from exhash import file_exhash
        p = host.check(path)
        try: cmds = _cmds(commands)
        except Exception as e: return f'could not parse commands: {agent_err(e)}'
        if not cmds: return 'no commands given'
        try: return clip(str(file_exhash(str(p), *cmds)))
        except Exception as e: return f'edit failed: {agent_err(e)}'

    def create_file(path: str, text: str = '') -> str:
        "Create (or overwrite) a whole file. For changes to an existing file prefer `edit_file`."
        try: return f'wrote {host.write(path, text)}'
        except Exception as e: return f'write failed: {agent_err(e)}'

    return [view_file, edit_file, create_file]

In [ ]:
#| export
def notebook_tools(host):
    "Notebooks, addressed by cell id rather than by line."

    def notebook_cells(path: str) -> str:
        "List a notebook's cells: id, type, and first line. Cell ids are what `edit_cell` addresses."
        try: rows = host.nb_cells(str(host.check(path)))
        except NotImplementedError: raise
        except Exception as e: return f'could not read notebook: {agent_err(e)}'
        return clip('\n'.join(f'{i}  {t:8} {(s or "").strip().splitlines()[0][:100] if (s or "").strip() else ""}'
                              for i, t, s in rows) or '(empty notebook)')

    def view_cell(path: str, cell_id: str) -> str:
        "Read one notebook cell as `lineno|hash|content` lines, ready to address with `edit_cell`."
        from exhash import lnhashview_cell
        try: return clip(str(lnhashview_cell(str(host.check(path)), cell_id)))
        except Exception as e: return f'could not read cell: {agent_err(e)}'

    def edit_cell(path: str, cell_id: str, commands: str) -> str:
        "Edit one notebook cell's source with exhash commands from `view_cell`. Same format as `edit_file`."
        from exhash import cell_exhash
        try: cmds = _cmds(commands)
        except Exception as e: return f'could not parse commands: {agent_err(e)}'
        try: return clip(str(cell_exhash(str(host.check(path)), cell_id, *cmds)))
        except Exception as e: return f'edit failed: {agent_err(e)}'

    def add_cell(path: str, source: str, index: int = -1, cell_type: str = 'code') -> str:
        "Insert a new cell into a notebook at `index` (-1 appends). Creates the notebook if needed."
        try: return f'added cell {host.nb_add_cell(str(host.check(path)), source, int(index), cell_type)} to {path}'
        except NotImplementedError: raise
        except Exception as e: return f'could not add cell: {agent_err(e)}'

    return [notebook_cells, view_cell, edit_cell, add_cell]

In [ ]:
#| export
def web_tools(host):
    "The web, for the questions whose answer depends on current documentation."

    def web_search(query: str) -> str:
        "Search the web. Returns titles and urls; follow up with `read_url` on the useful ones."
        docs = host.web_search(query, n=MAX_HITS)
        if not docs: return f'no results ({host.research_note})'
        return clip('\n'.join(f'{d.title}\n  {d.url}' for d in docs))

    def read_url(url: str) -> str:
        "Read one web page (or GitHub file, or arxiv paper) as markdown."
        d = host.read_url(url)
        return clip(d.text if d else f'could not read {url} ({host.research_note})')

    def research(query: str) -> str:
        "Search the web and read the top results into one cited digest. Slower than `web_search`; use for depth."
        return clip(host.research(query) or f'nothing found ({host.research_note})')

    return [web_search, read_url, research]

In [ ]:
#| export
def session_tools(host):
    "The live kernel the user is working in, and the terminal they are looking at."

    def list_vars() -> str:
        "List the variables visible in the user's live session: name, type, and a short value."
        return clip(host.list_vars() or '(empty session)')

    def run_python(code: str) -> str:
        """Run Python in the user's live kernel namespace.

        Read any variable freely; bind results to NEW names so they survive to the next
        call. Mutating or deleting the user's variables is refused -- rebind instead
        (`df2 = df.drop(...)`). Call `list_vars` first if you do not know what is there.
        """
        try: return clip(host.run_python(code))
        except NotImplementedError: raise
        except Exception as e: return f'run failed: {agent_err(e)}'

    def inspect_python(code: str, scope: str = 'isolated') -> str:
        """Look at the user's live variables by running Python that cannot change them.

        Two scopes. Both leave the user's variables exactly as they were; they differ in
        how much Python you get, so pick by what the question needs:

        - `scope='isolated'` (default) runs in an allowlist sandbox on a copy. Attribute
          reads and builtins work — `df.shape`, `len(df)`, `type(x).__name__` — and most
          library method calls are refused. Costs nothing to be wrong about.
        - `scope='overlay'` runs the real interpreter against the real namespace. Library
          calls work: `list(df.columns)`, `df.head(3).to_dict()`, `model.summary()`. Names
          you bind persist into your own layer for later calls. You still cannot delete,
          rebind or mutate anything the user made — that is refused, with an explanation.

        Start isolated; move to overlay when the sandbox refuses something you need. Neither
        needs approval, and both run while one of the user's cells is still going. For work
        that must land in the *user's* namespace, use `run_python` instead.
        """
        try: return clip(host.inspect_python(code, scope=scope))
        except NotImplementedError: raise
        except Exception as e: return f'inspection failed: {agent_err(e)}'

    def read_terminal(lines: int = 200) -> str:
        """Read what the IDE's terminal has printed -- a failing build, a stack trace, a test run.

        This is *read only*: it shows what the user ran, and cannot run anything. Use it
        when they mention an error they are looking at rather than asking them to paste it.
        """
        return clip(host.terminal_text(int(lines)) or 'the terminal has printed nothing yet')

    return [list_vars, run_python, inspect_python, read_terminal]

In [ ]:
#| export
def skill_tools(get_skills):
    "Reading a discovered skill's body. `get_skills` is a callable so a reload is picked up."

    def read_skill(name: str) -> str:
        """Read one skill in full: how to use a tool or a library that is already installed here.

        The skill list in your briefing gives names and one-line descriptions. Read the
        matching one *before* doing the work it describes, not after it has gone wrong.
        """
        from ramabana.skills import find
        ss = get_skills()
        s = find(ss, name)
        if s is None:
            return f'no skill matching {name!r}. Available: ' + ', '.join(x.name for x in ss)
        return clip(f'<skill name="{s.name}" from="{s.where}">\n{s.text()}\n</skill>', MAX_TOOL_CHARS * 3)

    return [read_skill]

In [ ]:
#| export
# ---------------------------------------------------------------------------
def tools_for(host, get_skills=None, extra=()):
    """Every tool this host can actually support, plus whatever extensions registered.

    Each group is probed with a harmless call and dropped whole if the host does not
    implement it. Whole groups rather than individual tools because the groups are the
    real units of capability: a host with no notebook representation cannot support any of
    the four notebook tools, and one with no kernel cannot support any of the session ones.
    """
    tools = []
    tools += code_tools(host)
    tools += file_tools(host)
    if _probe(host, lambda: host.nb_cells('.')): tools += notebook_tools(host)
    if _probe(host, lambda: host.web_search('', n=1)): tools += web_tools(host)
    if _probe(host, lambda: host.list_vars(), lambda: host.terminal_text(1)): tools += session_tools(host)
    if get_skills is not None and get_skills(): tools += skill_tools(get_skills)
    tools += list(extra or ())
    return tools

## Tests


In [ ]:
# A capability the host does not have is never offered. Telling a model about a tool that
# always fails is worse than never mentioning it: it keeps trying.
from ramabana.host import NullHost
from ramabana.testing import MemHost
names = sorted(t.__name__ for t in tools_for(NullHost(['/x'])))
print('NullHost offers:', names)
assert 'search_code' in names and 'view_file' in names
assert 'run_python' not in names and 'notebook_cells' not in names

In [ ]:
# MemHost can run python but has no `list_vars`/`terminal_text`, so the whole session group
# drops rather than arriving half-broken.
mem = sorted(t.__name__ for t in tools_for(MemHost()))
print('MemHost offers :', mem)
assert 'run_python' not in mem

In [ ]:
# The line an approval policy draws is exactly the set of tools that change something.
print('write tools:', sorted(WRITE_TOOLS))
assert WRITE_TOOLS == {'edit_file', 'create_file', 'edit_cell', 'add_cell', 'run_python'}

In [ ]:
# Results are clipped *here*, before the tokens are spent, not after.
long = 'x' * (MAX_TOOL_CHARS + 500)
out = clip(long)
print(out[-40:])
assert len(out) < len(long) and 'more chars' in out

In [ ]:
# The tools really do read and write through the host.
h = MemHost({'/proj/a.py': 'def a(): pass\n'})
ts = {t.__name__: t for t in tools_for(h)}
print(ts['view_file']('/proj/a.py'))
print(ts['search_code']('def a'))